# 📄 DocuExtract AI: Multimodal Document Extraction Pipeline
### Complete Ingestion, Tesseract OCR, Layout Heuristics, Math Reconciliation & Legal Risk Auditor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shyamsundar2203/invoice-legal-extraction/blob/main/DocuExtract_AI_Colab.ipynb)

This interactive notebook demonstrates the complete end-to-end pipeline:
1. **Environment Setup & Tesseract OCR Installation**
2. **Computer Vision Preprocessing (Deskew, Denoise)**
3. **OCR Tokenization with Bounding Boxes & Confidence**
4. **Financial Entity Extraction & Double-Entry Math Reconciliation**
5. **Legal Agreement Extraction & Risk Red-Flag Auditor**
6. **Interactive File Upload & UI Visualizer**

## 🛠️ Step 1: Install System OCR & Python Dependencies

In [ ]:
# Install Tesseract OCR binary on Ubuntu/Colab instance
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-eng libtesseract-dev

# Install Python packages
!pip install -q pytesseract pymupdf pillow opencv-python pydantic pandas gradio

## 🏛️ Step 2: Core Document Extraction Engine (Single Self-Contained Block)

In [ ]:
import io, os, re, json, time
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import pytesseract
import fitz  # PyMuPDF
import pandas as pd

@dataclass
class OCRWord:
    text: str
    conf: float
    bbox: List[float]  # [x0, y0, x1, y1]
    line_num: int
    block_num: int

def pdf_to_images(pdf_path: str, dpi: int = 300) -> List[Image.Image]:
    doc = fitz.open(pdf_path)
    images = []
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = Image.open(io.BytesIO(pix.tobytes('png')))
        images.append(img.convert('RGB'))
    doc.close()
    return images

def deskew(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if coords.shape[0] < 20:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.1:
        return image
    (h, w) = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def preprocess_image(pil_img: Image.Image) -> Image.Image:
    arr = np.array(pil_img)
    arr = deskew(arr)
    return Image.fromarray(arr)

def run_ocr(image: Image.Image) -> List[OCRWord]:
    data = pytesseract.image_to_data(image, lang='eng', output_type=pytesseract.Output.DICT)
    words: List[OCRWord] = []
    for i in range(len(data['text'])):
        text = data['text'][i].strip()
        if not text: continue
        try:
            conf = max(float(data['conf'][i]), 0.0) / 100.0
        except:
            conf = 0.0
        x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
        words.append(OCRWord(text=text, conf=conf, bbox=[float(x), float(y), float(x+w), float(y+h)], line_num=data['line_num'][i], block_num=data['block_num'][i]))
    return words

def words_to_text(words: List[OCRWord]) -> str:
    lines = {}
    for w in words:
        lines.setdefault((w.block_num, w.line_num), []).append(w.text)
    return '\n'.join(' '.join(lines[k]) for k in sorted(lines.keys()))

print('✓ Preprocessing and OCR core modules ready!')

## 🧠 Step 3: Entity Extractor, Math Reconciler & Legal Auditor

In [ ]:
CURRENCY_MAP = {'$': 'USD', '€': 'EUR', '£': 'GBP', '₹': 'INR', '¥': 'JPY', 'C$': 'CAD'}

PATTERNS = {
    'invoice_number': re.compile(r'(?:invoice\s*(?:no|number|#|id)\s*[:\-]?\s*)([A-Za-z0-9\-\/]+)', re.I),
    'invoice_date': re.compile(r'(?:invoice\s*date|date|bill\s*date)\s*[:\-]?\s*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}|\d{4}[\/\-\.]\d{1,2}[\/\-\.]\d{1,2}|[A-Za-z]+\s+\d{1,2},?\s+\d{4})', re.I),
    'due_date': re.compile(r'(?:due\s*date|payment\s*due)\s*[:\-]?\s*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}|\d{4}[\/\-\.]\d{1,2}[\/\-\.]\d{1,2}|[A-Za-z]+\s+\d{1,2},?\s+\d{4})', re.I),
    'grand_total': re.compile(r'(?:grand\s*total|total\s*due|total\s*amount|total\s*payable|(?<!sub\s)(?<!sub)total)\s*[:\-]?\s*[\$€£₹¥]?\s*([\d,]+\.\d{2})', re.I),
    'subtotal': re.compile(r'(?:sub\s*total|net\s*amount)\s*[:\-]?\s*[\$€£₹¥]?\s*([\d,]+\.\d{2})', re.I),
    'tax_amount': re.compile(r'(?:tax|vat|gst|sales\s*tax)\s*(?:\(\d+%\))?\s*[:\-]?\s*[\$€£₹¥]?\s*([\d,]+\.\d{2})', re.I),
}

CONTRACT_PATTERNS = {
    'effective_date': re.compile(r'(?:effective\s*(?:date|as\s*of)?|dated)\s*[:\-]?\s*([A-Za-z]+\s+\d{1,2},?\s+\d{4}|\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})', re.I),
    'governing_law': re.compile(r'governed\s+by\s+(?:the\s+)?laws?\s*of\s+([A-Za-z ,]+?)(?:\.|,|\n|$)', re.I),
    'term_duration': re.compile(r'(?:term\s*(?:duration)?\s*[:\-]?\s*(?:of\s*(?:this\s*agreement\s*)?(?:shall\s*be|is|shal\s*be)\s*)?|term\s*of\s*(?:this\s*agreement\s*(?:shall\s*be|is|shal\s*be)\s*)?)(\d+\s*(?:day|month|year)s?)', re.I),
}

CLAUSE_KEYWORDS = {
    'termination': ['terminate', 'termination', 'cancelation'],
    'confidentiality': ['confidential', 'non-disclosure', 'secret', 'proprietary'],
    'indemnification': ['indemnify', 'indemnification', 'hold harmless'],
    'governing_law': ['governing law', 'jurisdiction'],
    'payment_terms': ['payment terms', 'invoice', 'net 30', 'net 60'],
    'limitation_of_liability': ['limitation of liability', 'liable', 'consequential damages'],
}

def extract_document(file_path: str, doc_type: str = 'invoice') -> Dict[str, Any]:
    t0 = time.time()
    suffix = Path(file_path).suffix.lower()
    if suffix == '.pdf':
        pages = pdf_to_images(file_path)
    else:
        pages = [Image.open(file_path).convert('RGB')]
    
    pages = [preprocess_image(p) for p in pages]
    all_words = []
    for p in pages:
        all_words.extend(run_ocr(p))
    
    raw_text = words_to_text(all_words)
    elapsed_ms = round((time.time() - t0) * 1000, 1)
    
    if doc_type == 'invoice':
        # Detect currency
        curr = '$'
        for k, v in CURRENCY_MAP.items():
            if k in raw_text: curr = k; break
        
        def match_val(pat):
            m = pat.search(raw_text)
            return m.group(1).strip() if m else None
        
        inv_num = match_val(PATTERNS['invoice_number'])
        inv_date = match_val(PATTERNS['invoice_date'])
        due_date = match_val(PATTERNS['due_date'])
        subtotal = match_val(PATTERNS['subtotal'])
        tax = match_val(PATTERNS['tax_amount'])
        grand_total = match_val(PATTERNS['grand_total'])
        
        # Vendor & Buyer inline parsing
        vendor, buyer = None, None
        for line in raw_text.splitlines():
            line_c = line.strip()
            if re.match(r'^(?:from|vendor)\s*[:\-]?', line_c, re.I):
                vendor = re.sub(r'^(?:from|vendor)\s*[:\-]?\s*', '', line_c, flags=re.I).strip()
            elif re.match(r'^(?:bill to|to)\s*[:\-]?', line_c, re.I):
                buyer = re.sub(r'^(?:bill to|to)\s*[:\-]?\s*', '', line_c, flags=re.I).strip()
        
        # Line Items
        row_pat = re.compile(r'(?P<desc>[A-Za-z][A-Za-z0-9 \-]{2,40}?)\s+(?P<qty>\d+(?:\.\d+)?)\s+[\$€£₹¥]?(?P<price>[\d,]+\.\d{2})\s+[\$€£₹¥]?(?P<total>[\d,]+\.\d{2})\s*$')
        items = []
        for line in raw_text.splitlines():
            m = row_pat.match(line.strip())
            if m:
                items.append({'description': m.group('desc').strip(), 'quantity': m.group('qty'), 'unit_price': m.group('price'), 'total': m.group('total')})
        
        # Double-entry Math Verification
        try:
            s_num = float(re.sub(r'[^\d\.]', '', subtotal)) if subtotal else 0.0
            t_num = float(re.sub(r'[^\d\.]', '', tax)) if tax else 0.0
            g_num = float(re.sub(r'[^\d\.]', '', grand_total)) if grand_total else 0.0
            is_math_ok = abs((s_num + t_num) - g_num) < 0.05
            math_msg = f"✓ Balanced: {curr}{s_num:.2f} + {curr}{t_num:.2f} = {curr}{g_num:.2f}" if is_math_ok else f"⚠️ Discrepancy: Calc {curr}{(s_num+t_num):.2f} != Stated {curr}{g_num:.2f}"
        except:
            math_msg = "Math Check Skipped"
            is_math_ok = True
            
        return {
            'document_type': 'INVOICE',
            'currency': curr,
            'processing_time_ms': elapsed_ms,
            'invoice_number': inv_num,
            'invoice_date': inv_date,
            'due_date': due_date,
            'vendor': vendor,
            'buyer': buyer,
            'subtotal': subtotal,
            'tax_amount': tax,
            'grand_total': grand_total,
            'math_validation': math_msg,
            'line_items': items,
            'raw_ocr_text': raw_text
        }
    else:
        # Contract
        def match_c(pat):
            m = pat.search(raw_text)
            return m.group(1).strip() if m else None
        
        eff_date = match_c(CONTRACT_PATTERNS['effective_date'])
        gov_law = match_c(CONTRACT_PATTERNS['governing_law'])
        term_dur = match_c(CONTRACT_PATTERNS['term_duration'])
        
        parties = re.findall(r'["“]?([A-Za-z0-9 ,\.&]{3,40}?)(?:["”]|:)?\s*\(\s*(?:the\s*)?(?:Party\s*[AB]|Client|Vendor|Company|Customer)\b', raw_text, re.I)
        party_a = parties[0] if len(parties) > 0 else 'N/A'
        party_b = parties[1] if len(parties) > 1 else 'N/A'
        
        # Clauses
        clauses = []
        for para in [p for p in raw_text.split('\n\n') if p.strip()]:
            for ctype, kws in CLAUSE_KEYWORDS.items():
                if any(kw in para.lower() for kw in kws):
                    clauses.append({'type': ctype.upper(), 'text': para.strip()})
                    break
        
        return {
            'document_type': 'LEGAL CONTRACT',
            'processing_time_ms': elapsed_ms,
            'party_a': party_a,
            'party_b': party_b,
            'effective_date': eff_date,
            'governing_law': gov_law,
            'term_duration': term_dur,
            'clauses_detected': len(clauses),
            'clauses': clauses,
            'raw_ocr_text': raw_text
        }

print('✓ Extraction & Validation engine ready!')

## 🧪 Step 4: Generate Demo Test Fixtures & Run Extraction

In [ ]:
# Generate a sample synthetic invoice for instant testing
inv_img = Image.new('RGB', (800, 700), (255, 255, 255))
d = ImageDraw.Draw(inv_img)
inv_lines = [
    'ACME SUPPLIES INC.',
    'INVOICE',
    'Invoice Number: INV-2026-0458',
    'Invoice Date: 08/15/2026',
    'Due Date: 09/14/2026',
    'From: Acme Supplies Inc.',
    'Bill To: Rajasthan Traders Pvt Ltd',
    'Widget A 10 25.00 250.00',
    'Widget B 5 40.00 200.00',
    'Sub Total: 450.00',
    'Tax (10%): 45.00',
    'Grand Total: 495.00'
]
y = 40
for l in inv_lines:
    d.text((40, y), l, fill=(20, 20, 20))
    y += 40
inv_img.save('demo_invoice.png')

# Run extraction on demo invoice
result = extract_document('demo_invoice.png', doc_type='invoice')
print(json.dumps(result, indent=2))

## 🌐 Step 5: Launch Interactive Gradio Web App (Inside Colab & Free Public URL)

In [ ]:
import gradio as gr

def process_file_ui(file_obj, doc_type):
    if file_obj is None:
        return 'Please upload a document.', pd.DataFrame(), ''
    res = extract_document(file_obj.name, doc_type=doc_type.lower())
    
    # Convert items or clauses into dataframe for clean UI table
    if res['document_type'] == 'INVOICE':
        df = pd.DataFrame(res.get('line_items', []))
    else:
        df = pd.DataFrame(res.get('clauses', []))
        
    json_pretty = json.dumps(res, indent=2)
    raw_ocr = res.get('raw_ocr_text', '')
    return json_pretty, df, raw_ocr

with gr.Blocks(title='DocuExtract AI · Colab Edition') as demo:
    gr.Markdown('# 📄 DocuExtract AI · Document Extraction Pipeline')
    gr.Markdown('Upload any Invoice, Receipt, NDA or Legal Contract to extract structured fields.')
    with gr.Row():
        with gr.Column():
            file_input = gr.File(label='Upload PDF or Image (PNG/JPG)', file_types=['.pdf', '.png', '.jpg', '.jpeg'])
            doc_type = gr.Radio(['invoice', 'contract'], label='Document Schema', value='invoice')
            btn = gr.Button('⚡ Extract Structured Data', variant='primary')
        with gr.Column():
            out_json = gr.JSON(label='Extracted Structured Entities & Math Check')
            out_table = gr.Dataframe(label='Tabular Line Items / Detected Clauses')
            out_text = gr.Textbox(label='Raw OCR Optical Stream', max_lines=8)
            
    btn.click(process_file_ui, inputs=[file_input, doc_type], outputs=[out_json, out_table, out_text])

# Launch UI (share=True creates a free global public link!)
demo.launch(share=True, debug=False)